Red Neuronal Simple

In [1]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import os
import kagglehub
from torchsummary import summary
import torch.nn.functional as F
import csv

C:\Users\Daniel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creación del dataset y dataloader para el entrenamiento.

In [3]:
# --- FRAGMENTO NÚMERO 2 ---
print("\n--- Cargando Datasets y DataLoaders desde los CSVs generados ---")

datasets = {}
dataloaders = {}

IMAGE_SIZE = (416, 416)
BATCH_SIZE = 32

KAGGLE_DATASET_ID = 'pkdarabi/cardetection'
KAGGLE_DOWNLOAD_PATH = kagglehub.dataset_download(KAGGLE_DATASET_ID)
base_data_path = os.path.join(KAGGLE_DOWNLOAD_PATH, "car")
DATA_DIR = './'
DIVISIONES = ["train", "valid", "test"]

# Normalización OBLIGATORIA para modelos pre-entrenados
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

# 1. Transformaciones para ENTRENAMIENTO (con aumento de datos)
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5), # Volteo aleatorio
    transforms.RandomRotation(15),           # Rotación aleatoria
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), # Cambios de color
    transforms.ToTensor(),                   # Convertir a Tensor
    normalize,                               # Normalizar
])

# 2. Transformaciones para VALIDACIÓN y TEST (solo limpiar)
val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    normalize,
])

# Esta clase ahora funcionará porque leerá los CSVs que acabamos de crear.
class DataSet(Dataset):
    def __init__(self, archivo_csv, directorio_imagenes, transform=None):
        self.directorio_imagenes = directorio_imagenes
        self.transform = transform
        
        try:
            self.full_labels_df = pd.read_csv(archivo_csv)
        except FileNotFoundError:
            print(f"  - ¡Error! No se encontró el archivo CSV: {archivo_csv}")
            self.full_labels_df = pd.DataFrame(columns=['nombre_archivo'])
            
        self.imagenes_unicas = self.full_labels_df['nombre_archivo'].unique()
        self.labels_grouped = self.full_labels_df.groupby('nombre_archivo')

    def __len__(self):
        return len(self.imagenes_unicas)

    def __getitem__(self, idx):
        image_name = self.imagenes_unicas[idx]
        image_path = os.path.join(self.directorio_imagenes, image_name)
        
        try:
            image = Image.open(image_path).convert('RGB')
        except FileNotFoundError:
            return torch.zeros(3, IMAGE_SIZE[0], IMAGE_SIZE[1]), torch.tensor(-1) 

        boxes_df = self.labels_grouped.get_group(image_name)
        clase_idx = int(boxes_df['clase_indice'].iloc[0])
        label = torch.tensor(clase_idx, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label
 
for label in DIVISIONES:
    # Construir rutas
    ruta_csv = os.path.join(DATA_DIR, f"CSVs/{label}_corregido.csv") 
    ruta_imgs = os.path.join(base_data_path, label, "images")
    
    print(f"\nProcesando conjunto de datos: {label}")
    print(f"Ruta del CSV: {ruta_csv}")
    print(f"Ruta de las imágenes: {ruta_imgs}")

    if not os.path.exists(ruta_csv) or not os.path.exists(ruta_imgs):
        print("  - Error: No se encontró el CSV o el directorio de imágenes.")

    else:
        # Asignar la transformación correcta según el split
        if label == 'train':
            transform_actual = train_transform
        else:
            transform_actual = val_transform
            
        # Crear el Dataset
        dataset = DataSet(archivo_csv=ruta_csv, 
                          directorio_imagenes=ruta_imgs, 
                          transform=transform_actual)
        
        if len(dataset) > 0:
            print(f"  - Tipo de objeto creado: {type(dataset)}")
            print(f"  - Número total de imágenes: {len(dataset)}")

            datasets[label] = dataset
            
            # --- OPTIMIZACIÓN DE GPU ---
            dataloaders[label] = DataLoader(
                dataset,
                batch_size=BATCH_SIZE,
                shuffle=(label == 'train'),
                pin_memory=True,            # Acelera la transferencia a GPU
            )
            print(f"  - DataLoader para '{label}' creado (OPTIMIZADO).")

        else:
            print("  - Error: El dataset está vacío (CSV vacío o no se pudo leer).")

print("\n--- Proceso de carga completado ---")


--- Cargando Datasets y DataLoaders desde los CSVs generados ---

Procesando conjunto de datos: train
Ruta del CSV: ./CSVs/train_corregido.csv
Ruta de las imágenes: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5\car\train\images
  - Error: No se encontró el CSV o el directorio de imágenes.

Procesando conjunto de datos: valid
Ruta del CSV: ./CSVs/valid_corregido.csv
Ruta de las imágenes: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5\car\valid\images
  - Error: No se encontró el CSV o el directorio de imágenes.

Procesando conjunto de datos: test
Ruta del CSV: ./CSVs/test_corregido.csv
Ruta de las imágenes: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5\car\test\images
  - Error: No se encontró el CSV o el directorio de imágenes.

--- Proceso de carga completado ---


Creación de la red neuronal.

In [4]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(3*416*416, 512)
        self.fc2 = nn.Linear(512, 256)     # Capa oculta con 256 neuronas
        self.fc3 = nn.Linear(256, 128)     # Capa oculta con 128 neuronas
        self.fc4 = nn.Linear(128, 14)      # Capa de salida con 14 clases (0-13)
        self.activation = nn.ReLU()        # Función de activación Sigmoide

    def forward(self, x):
        # x = x.view(-1, 416*416)            
        x = x.view(x.size(0), -1)       # Aplanar la imagen de 416x416 a un vector de 173056
        #print(x.shape)                  # Mostrar la forma del tensor después de aplanarlo
        x = self.fc1(x)                
        x = self.activation(x)            # Función de activación ReLU en la capa oculta
        #print(x.shape)                   # Mostrar la forma del tensor después de la primera capa
        x = self.fc2(x)                  
        x = self.activation(x)
        x = self.fc3(x)
        x = self.activation(x)
        x = self.fc4(x)
        print(x.shape)                   # Mostrar la forma del tensor después de la segunda capa
        return x

In [5]:
model = SimpleNN()
summary(model, (3, 416, 416)) # Resumen del modelo
# El modelo y las dimension de entrad de los datos


torch.Size([2, 14])
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 512]     265,814,528
              ReLU-2                  [-1, 512]               0
            Linear-3                  [-1, 256]         131,328
              ReLU-4                  [-1, 256]               0
            Linear-5                  [-1, 128]          32,896
              ReLU-6                  [-1, 128]               0
            Linear-7                   [-1, 14]           1,806
Total params: 265,980,558
Trainable params: 265,980,558
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 1.98
Forward/backward pass size (MB): 0.01
Params size (MB): 1014.64
Estimated Total Size (MB): 1016.63
----------------------------------------------------------------


Entrenamiento de la red.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNN().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.5)

EPOCHS = 10
for epoch in range(EPOCHS):
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        labels_one_hot = F.one_hot(labels, num_classes=15).float()  # 15 clases
        loss = criterion(outputs, labels_one_hot)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"[Epoch {epoch + 1}] loss: {running_loss / len(train_loader):.3f}")
